# Rumo ao Desconhecido: Tratando Drift em ML
## Parte I — O sintético como suíte de testes do detector

**Python Brasil 2026 · Blocos I e II-A · ~75 min**

---

### Como usar este notebook

| símbolo | significa |
|---|---|
| 🛠️ **VOCÊ IMPLEMENTA** | célula com `TODO`. O teste logo abaixo é o seu gabarito. |
| 🎲 **APOSTE** | pare, levante a mão, **depois** rode. Custa 90s e você não esquece. |
| ✅ **CHECKPOINT** | autoverificação. Se der ❌, levante o cartão 🔴. |
| 📖 **Aprofundamento** | leia em casa. Em sala, eu falo isso. |

> Travou? Abra `01_sintetico_SOLUCAO.ipynb`, copie a célula e siga.
> **Ninguém fica para trás por causa de um exercício.**

---

### A tese, em uma frase

> Um detector de drift é um **instrumento de medição**. E ninguém valida um
> instrumento contra uma amostra de valor desconhecido — você calibra a
> balança com um peso padrão de 1 kg, e só depois pesa o que não conhece.

**Este notebook é o peso padrão.** O Bosch, no notebook 02, é o que não conhecemos.

In [ ]:
# ============================================================
#  CÉLULA 0 — SETUP (idempotente: rode de novo se o runtime cair)
# ============================================================
!curl -sSL https://raw.githubusercontent.com/arcursino/python-br-2026/main/setup_colab.py -o /tmp/setup_colab.py
%run /tmp/setup_colab.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from driftkit.notebook import banner, requer, checkpoint, aposte, resposta, cli
from driftkit.data import dir_dados

DATA = dir_dados()

# Fontes de PALCO, não de notebook. Gráfico que você não lê da fila 12
# não é evidência — é decoração.
plt.rcParams.update({
    "font.size": 14, "axes.titlesize": 18, "axes.labelsize": 15,
    "legend.fontsize": 13, "xtick.labelsize": 13, "ytick.labelsize": 13,
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25,
})
pd.set_option("display.width", 160)

checkpoint(1, ambiente_ok=True, driftkit_importado=True)

---
# 🧭 Bloco I — Fundamentos (15 min)

## I.1 O modo de falha que define o problema

Um serviço web quebrado devolve **HTTP 500**. Um modelo degradado devolve
**`200 OK`** com uma previsão errada, com a mesma confiança de sempre,
indefinidamente.

Ninguém abre incidente porque nada *parece* quebrado. O custo aparece semanas
depois, em refugo, retrabalho e garantia de campo.

## I.2 A decomposição que organiza tudo

$$P(X, Y) = P(Y \mid X)\,P(X)$$

| Termo que muda | Nome | Exemplo na linha de montagem |
|---|---|---|
| $$P(X)$$ | **data drift** | campanha muda o mix de aro |
| $$P(Y \mid X)$$ | **concept drift** | transdutor descalibra: a feature passa a **mentir** |
| $$P(Y)$$ | **label shift** | novo critério de aprovação da inspeção |
| schema, unidade | **data quality** | tag do PLC volta a publicar em kPa |

### A assimetria que quase ninguém internaliza

- **Data drift NÃO implica perda de performance.** Se $$P(Y\mid X)$$ está intacto,
  a fronteira de decisão continua correta.
- **Concept drift implica perda de performance, sempre.** A função que o modelo
  aprendeu deixou de descrever o mundo.

> Monitore $$P(X)$$ porque é **barato** e chega **antes do rótulo**.
> Decida com base em $$P(Y\mid X)$$ e nos KPIs de negócio.

<details>
<summary>📖 <b>Aprofundamento: taxonomia temporal, a pirâmide de 4 camadas e o problema epistemológico</b> — leia em casa</summary>

### Taxonomia temporal (Gama et al., 2014)

| Padrão | Assinatura | Exemplo industrial | Ação |
|---|---|---|---|
| **Súbito** | degrau | troca de lote, firmware novo | retreino / correção |
| **Gradual** | rampa lenta | **desgaste de ferramenta, deriva de calibração** | o mais comum e mais insidioso |
| **Recorrente** | ciclo | turno C, variação térmica | **modelar, não combater** |
| **Blip** | pico de 1 janela | falha pontual de rede | ignorar — retreinar aqui é o erro mais caro |

A segunda linha é a que mais aparece em manufatura e a que menos aparece em
tutoriais. A terceira é a fonte nº 1 de falso positivo: sazonalidade tratada
como drift gera **retreino perpétuo**.

### A pirâmide de monitoramento

| # | Camada | O que mede | Latência | Precisa de rótulo? |
|---|---|---|---|---|
| 1 | Integridade | nulos, schema, unidades, ranges | segundos | não |
| 2 | Distribuição | KS, χ², PSI, presença | minutos | não |
| 3 | Performance | PR-AUC, MCC, log-loss | dias–semanas | **sim** |
| 4 | KPI de negócio | FTT, PPM, Cpk, garantia | semanas | sim |

A camada 2 é um **proxy antecipado** da camada 3 — você a monitora porque o
rótulo ainda não chegou. E existe drift que **só** aparece na camada 3.

### O problema epistemológico

Seu monitor alarmou no período 14 de um dado real. Três perguntas:

1. Houve drift, ou o limiar está apertado demais?
2. **Quando começou?** (o alarme é o instante da detecção, não do evento)
3. Quantos eventos ele **deixou passar**?

Em dado real, **nenhuma das três tem resposta** — e as duas métricas com que se
avalia um detector (atraso de detecção e taxa de falso positivo) são
indefiníveis sem ground truth.

> Sem calibração prévia, um monitor de drift é um **gerador de opinião com
> aparência de gráfico**.

**A saída é plantar a falha.**

</details>

In [ ]:
aposte(
    "Vou criar dois cenários. Em (A) as distribuições das features mudam MUITO. "
    "Em (B) elas ficam praticamente IDÊNTICAS. Em qual dos dois o modelo vai "
    "errar mais?",
    ("A", "B", "nos dois igual"),
)

In [ ]:
# ============================================================
#  O experimento fundamental. Todo o tutorial sai daqui.
# ============================================================
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
X_tr = rng.normal(0, 1, (4000, 2))
y_tr = (X_tr[:, 0] + X_tr[:, 1] > 0).astype(int)
m = LogisticRegression().fit(X_tr, y_tr)

# (A) DATA DRIFT — P(X) muda muito, P(y|X) intacto
X_a = rng.normal(1.5, 1.6, (4000, 2))
y_a = (X_a[:, 0] + X_a[:, 1] > 0).astype(int)

# (B) CONCEPT DRIFT — P(X) idêntico, P(y|X) invertido no segundo eixo
X_b = rng.normal(0, 1, (4000, 2))
y_b = (X_b[:, 0] - X_b[:, 1] > 0).astype(int)

for nome, Xn, yn in [("A) data drift   ", X_a, y_a), ("B) concept drift", X_b, y_b)]:
    ks = max(ks_2samp(X_tr[:, j], Xn[:, j]).statistic for j in range(2))
    auc = roc_auc_score(yn, m.predict_proba(Xn)[:, 1])
    print(f"{nome} | KS máx nas features = {ks:.3f} | AUC = {auc:.3f}")

print("\nA: o detector de distribuição GRITA  → e o modelo está PERFEITO.")
print("B: o detector de distribuição fica MUDO → e o modelo virou MOEDA.")
print("\n>>> Se você apostou em A, você está em boa companhia. E errado.")

---
## 🗣 Atividade 1 — Classifique os cinco casos (12 min, em duplas)

Para cada caso: **(a)** que tipo de drift? **(b)** qual camada detecta primeiro?
**(c)** retreinar resolve?

| # | Caso na linha Tire & Wheel |
|---|---|
| 1 | A balanceadora da CEL-03 teve o rolamento trocado. O refugo subiu, mas **todas as features seguem na faixa histórica**. |
| 2 | Após atualização do supervisório, `pressao_psi` chega multiplicada por 6.895. |
| 3 | Campanha de 30 dias com 80% de aro R20 para um lançamento. O PSI explode em três features. |
| 4 | Novo fornecedor entra com 25% do volume. O modelo nunca viu essa categoria. |
| 5 | A taxa de falha do turno C é sistematicamente maior. **Há dois anos.** |

**Em duplas, 8 minutos.** Concentrem-se nos casos **2, 3 e 5** — e reparem que
são justamente os três em que *retreinar é a decisão errada*.

Só rode a célula abaixo quando eu disser. 🙂

In [ ]:
resposta("atividade1")

---
# 🔬 Bloco II-A — O sintético como suíte de testes (45 min)

## A inversão de perspectiva

Até agora você provavelmente pensava assim:
*"tenho um dataset, quero detectar drift nele."*

Inverta:

> ## O **detector** é o produto.
> ## O **dataset sintético** é o teste unitário.

Se o detector é código que vai rodar em produção todo dia — parando linha e
abrindo ordem de serviço — então ele exige o mesmo rigor de qualquer código
crítico: **testes com resultado esperado conhecido**.

| Conceito de teste | Equivalente aqui |
|---|---|
| fixture | o dataset sintético com drift plantado |
| caso de teste (TC) | um evento: campanha, offset de sensor, troca de unidade |
| asserção | "o monitor **deve** alarmar" / "**não deve** alarmar" |
| **controle negativo** | regime estável: exigimos **silêncio** |
| suíte de regressão | roda no CI a cada mudança de limiar |

### A linha do tempo da fixture

| dias | TC | evento | P(X) | Performance | Ação correta |
|---|---|---|---|---|---|
| 1–29 | — | referência | — | — | — |
| 30–59 | **TC-1** | operação normal | 🤫 | ✅ | nada (controle negativo) |
| 60–89 | **TC-2** | campanha aro R20 | 🚨 | ✅ | registrar, **não** retreinar |
| 90–119 | **TC-3** | offset +10 Nm na CEL-02 | 🤫 | 💥 | **metrologia**, não retreino |
| 120–124 | TC-3' | ferramenta recalibrada | 🤫 | ✅ | encerrar incidente |
| 125–130 | **TC-4** | tag do PLC em kPa | ⚠️ | 📉 | corrigir pipeline |

**TC-2 × TC-3 é a prova cruzada:** exigem respostas **opostas**. Um detector que
responde igual às duas é cego — não importa a estatística por trás.

In [ ]:
# ============================================================
#  Passo 0: testar a FIXTURE, antes de acreditar em qualquer detector.
#  Fixture errada faz detector CORRETO parecer errado.
# ============================================================
from driftkit.fixtures import (
    NUM, CAT, RANGES, EVENTOS, TORQUE_ALVO, TORQUE_LSL, TORQUE_USL,
    gerar_fixture, janela, validar_fixture, cpk, ppm_fora_spec,
)

banner("Fixture Tire & Wheel", "130 dias · 4 células · drift plantado com gabarito")

SEED = 7
df = gerar_fixture(seed=SEED)
ref = df.query("dia < 30")

print(f"{len(df):,} peças × {df.shape[1]} colunas | {df.dia.nunique()} dias")
print(f"prevalência de falha na referência: {ref.falha.mean():.1%}\n")

checks = validar_fixture(df)
print(checks.to_string(index=False))
falhas_fixture = checks.query("status == 'FAIL'")
assert falhas_fixture.empty, "a fixture está errada — nada abaixo teria significado"
print("\n✅ a fixture contém exatamente o que promete.")

### 🔎 O mecanismo do TC-3 — leia com atenção, é o coração do tutorial

```python
torque_real   ~ N(110, 1.15)      # a FÍSICA não mudou
torque_medido = torque_real - 10  # o SENSOR mente (lê 10 Nm a menos)
```

O operador vê 100 Nm, acha que está frouxo, e a apertadeira compensa. O torque
real vai a ~120 Nm — **fora da especificação superior**, gerando trinca no cubo.

Consequência: `torque_medido` continua com distribuição *parecida* com a de
referência (média deslocada de 10 num range de ~40), mas o mapeamento
feature → falha **inverteu**. Isso é concept drift puro. Nenhum teste sobre
$$P(X)$$ tem **obrigação matemática** de pegar isso.

---
## 🛠️ VOCÊ IMPLEMENTA #1 — a camada mais barata da pirâmide (5 min)

`viola_range` é a **camada 1**: fração de registros fora da especificação de
engenharia. Custo `O(n)`, sem referência, sem rótulo, sem modelo — e é a única
camada que pega troca de unidade, o caso em que retreinar grava o erro no modelo.

**Três linhas bastam.** O teste abaixo é o seu gabarito.

In [ ]:
def viola_range(cur: pd.DataFrame, ranges: dict) -> dict[str, float]:
    """Fração de registros fora do range de engenharia, por feature.

    Parameters
    ----------
    cur : DataFrame da janela atual
    ranges : {feature: (limite_inferior, limite_superior)}

    Returns
    -------
    dict {feature: fracao_fora} — só para features presentes em `cur`.
    """
    fora = {}
    # TODO: para cada feature em `ranges` que exista em `cur`,
    #       calcule a fração de valores < lo ou > hi.
    #       Dica: ((s < lo) | (s > hi)).mean()
    raise NotImplementedError("implemente e rode a célula seguinte")
    return fora

In [ ]:
from driftkit.testing import checar_viola_range

checar_viola_range(viola_range)

In [ ]:
# ============================================================
#  REBIND — rode SEMPRE, mesmo se o exercício acima falhou.
# ============================================================
# `viola_range` volta a ser usada na PROVA CRUZADA, ~10 células abaixo. Sem
# este rebind, quem não terminou o exercício trava no meio do clímax com um
# NotImplementedError longe da célula que o causou — e `res` nunca é criado,
# derrubando o checkpoint final em cascata.
#
# Isto é uma célula SEPARADA de propósito: dentro da célula do `checar_*`, o
# import nunca roda quando o check levanta a exceção.
from driftkit.detectors import viola_range   # noqa: F811
print("→ `viola_range`: versão oficial e testada ativa a partir daqui.")

---
## 🛠️ VOCÊ IMPLEMENTA #2 — o predicado que separa monitor de gerador de spam (6 min)

Quando um monitor deve alarmar? Você tem quatro grandezas:

- `p_valor` — significância do teste (KS, χ²)
- `efeito` — magnitude (PSI, Wasserstein)
- `alpha_corrigido` — nível já dividido por nº de features (**Bonferroni**)
- `psi_limiar` — piso de magnitude

**A pergunta que vale o bloco inteiro: é `E` ou é `OU`?**

Pense antes de escrever. Com 50 mil peças/dia, **todo** p-valor é zero.

In [ ]:
def alarma(p_valor: float, efeito: float, alpha_corrigido: float, psi_limiar: float) -> bool:
    """Decide se esta feature entra em estado de alarme.

    Atenção: `p_valor` e `efeito` podem vir como np.nan (não mensurável).
    NaN em comparação devolve False silenciosamente — pense se isso é
    o que você quer, e o que significa.
    """
    # TODO: uma linha.
    raise NotImplementedError

In [ ]:
from driftkit.testing import checar_predicado_alarme

checar_predicado_alarme(alarma)

In [ ]:
# ============================================================
#  REBIND — idem. Célula própria, roda mesmo com o exercício em aberto.
# ============================================================
from driftkit.detectors import alarma   # noqa: F811
print("→ `alarma`: versão oficial ativa.")

---
## Construindo o instrumento

Agora usamos a versão oficial, que vive em `src/driftkit/detectors.py` — testada,
importável, e a mesma que roda no CI. Repare em `frozen=True`:

> **A referência não pode mudar depois da construção.**

Isso não é preciosismo de engenharia. Referência que se move sozinha é a causa
nº 1 de detector cego a deriva lenta — o sapo que cozinha devagar. Provamos isso
com um teste em `tests/test_limitacoes.py`.

In [ ]:
from driftkit.detectors import DriftDetector
from driftkit.modelo import treinar, metricas

requer("df", "ref")

detector_bruto = DriftDetector.from_reference(
    ref, num=list(NUM), cat=list(CAT), ranges=RANGES,
    psi_limiar=0.10,      # ⚠️ o "limiar da indústria" — vamos DESMENTIR em 2 células
)
modelo = treinar(ref, num=NUM, cat=CAT)

print(detector_bruto.report(janela(df, 45)).resumo())

In [ ]:
aposte(
    "O PSI > 0.1 é a regra de ouro que aparece em todo blog post de MLOps. "
    "Vou dividir a referência ao meio 40 vezes e medir o PSI de dado SEM "
    "NENHUM DRIFT contra si mesmo. O piso de ruído deste detector vai ficar "
    "acima ou abaixo de 0.1?",
    ("acima", "abaixo", "exatamente 0.1"),
)

In [ ]:
# ============================================================
#  TESTE A/A — o passo que quase ninguém faz, e que muda tudo
#  Dividimos a referência ao meio. Não há drift nenhum.
#  Logo: TUDO que aparecer aqui é ruído do próprio instrumento.
# ============================================================
banner("Teste A/A", "medindo o piso de ruído do detector", icone="📏")

piso = detector_bruto.calibrar_piso(n_repeticoes=40, seed=SEED)
for k, v in piso.items():
    print(f"  {k:<24} {v}")

PISO_AA = piso["piso_aa"]
PSI_ALARME = piso["psi_alarme_sugerido"]

print(f"\n  O 0.1 do blog post é {0.1 / max(PISO_AA, 1e-9):.0f}× o piso real deste detector.")
print(f"  Nesta fixture ele é conservador demais: deixaria passar drift real.")
print(f"  Em 160 features e 40 mil amostras/janela (notebook 02) ele será o oposto:")
print(f"  vai gerar alarme perpétuo. Guarde este número — vamos comparar.\n")
print(f"  >>> limiar CALIBRADO = {PSI_ALARME:.5f}  (3× o piso medido)")

# o detector definitivo, agora com limiar medido e não herdado
detector = detector_bruto.com(psi_limiar=PSI_ALARME)

> ### 🎯 A lição que vale a inscrição no tutorial
>
> **Não existe limiar universal.** O piso de ruído depende do número de
> features, do tamanho da janela e da distribuição dos dados. Você **mede** o
> seu com um teste A/A; não herda o de um blog post.
>
> E o teste A/A é a coisa mais barata deste notebook: 15 linhas, 20 segundos.

In [ ]:
aposte(
    "Agora a prova cruzada. TC-2 é uma campanha de aro R20 (o mix muda muito). "
    "TC-3 é o transdutor com offset de +10 Nm. Em qual deles o PSI vai passar "
    "do limiar? E em qual o modelo vai DE FATO errar mais?",
    ("TC-2 nos dois", "TC-3 nos dois", "cruzado"),
)

In [ ]:
# ============================================================
#  A PROVA CRUZADA — leia a DIAGONAL desta tabela
# ============================================================
requer("detector", "modelo")

linhas = []
for tc, dia in [("TC-1 estável", 45), ("TC-2 covariate", 80),
                ("TC-3 concept", 110), ("TC-3' recalibr.", 122), ("TC-4 unidade", 128)]:
    w = janela(df, dia, w=3 if dia != 128 else 2)
    seg = w.query("equipamento == 'CEL-02'")
    rel = detector.report(w)
    m_g = metricas(modelo, w, num=NUM, cat=CAT)
    m_s = metricas(modelo, seg, num=NUM, cat=CAT)
    fora = max(viola_range(w, RANGES).values(), default=0.0)
    linhas.append({
        "caso": tc, "dia": dia,
        "n_drift": rel.n_drift,
        "efeito_max": round(rel.efeito_max, 4),
        "x_limiar": round(rel.efeito_max / detector.psi_limiar, 1),
        "fora_range": f"{fora:.0%}",
        "AUC_global": m_g.roc_auc,
        "AUC_CEL02": m_s.roc_auc,
    })

res = pd.DataFrame(linhas)
print(res.to_string(index=False))

# --- prosa DERIVADA da execução. Nenhum número digitado à mão. ---
from IPython.display import Markdown

r2 = res.query("caso.str.startswith('TC-2')").iloc[0]
r3 = res.query("caso.str.startswith('TC-3 ')").iloc[0]
r1 = res.query("caso.str.startswith('TC-1')").iloc[0]

Markdown(f"""
### Leia a diagonal

- **TC-2** (campanha de R20): efeito **{r2.efeito_max:.3f}** — cerca de
  **{r2.x_limiar:.0f}× o limiar calibrado** — e a AUC global vai de
  {r1.AUC_global:.3f} para **{r2.AUC_global:.3f}**
  (Δ = **{r2.AUC_global - r1.AUC_global:+.3f}**).
  *O monitor grita e não há dano nenhum.*

- **TC-3** (offset de +10 Nm): efeito **{r3.efeito_max:.3f}** —
  **{r3.x_limiar:.1f}× o limiar** — e a AUC da CEL-02 cai de
  {r1.AUC_CEL02:.3f} para **{r3.AUC_CEL02:.3f}**
  (Δ = **{r3.AUC_CEL02 - r1.AUC_CEL02:+.3f}**).
  *O monitor está calmo e o modelo está {'pior que uma moeda' if r3.AUC_CEL02 < 0.5 else 'próximo de uma moeda'}.*

> **Duas situações que exigem respostas opostas.** Um pipeline que dispara
> retreino automático nas duas é um pipeline perigoso — e é o pipeline padrão
> do mercado.
""")

In [ ]:
# ============================================================
#  O experimento no tempo — e a honestidade sobre o monitor global
# ============================================================
trilha = []
# passo 2: 50 pontos em vez de 99. Gráfico idêntico, e esta é a célula mais
# cara do notebook (2 métricas + 1 report por dia).
for dia in range(32, 131, 2):
    w = janela(df, dia)
    seg = w.query("equipamento == 'CEL-02'")
    m_g = metricas(modelo, w, num=NUM, cat=CAT)
    m_s = metricas(modelo, seg, num=NUM, cat=CAT)
    trilha.append({"dia": dia, "auc_global": m_g.roc_auc, "auc_cel02": m_s.roc_auc,
                   "efeito": detector.report(w).efeito_max})
trilha = pd.DataFrame(trilha)

fig, (ax, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True,
                              gridspec_kw={"height_ratios": [2, 1]})

ax.axhspan(0.40, 0.55, color="red", alpha=0.07)
ax.plot(trilha.dia, trilha.auc_global, "k-", lw=3.5, label="AUC global (a linha inteira)")
ax.plot(trilha.dia, trilha.auc_cel02, "r--", lw=3.5, label="AUC da CEL-02 (segmentado)")
ax.axhline(0.5, color="gray", ls=":", lw=1.5)
ax.set_ylabel("ROC-AUC")
ax.set_title("Catastrófico na célula, discreto na linha")
ax.legend(loc="lower left")
for x, txt in [(60, "TC-2\ncampanha R20"), (90, "TC-3\noffset +10 Nm"),
               (120, "recalibrado"), (125, "TC-4\nkPa")]:
    for a in (ax, ax2):
        a.axvline(x, color="gray", ls="--", lw=1.2, alpha=0.7)
    ax.annotate(txt, xy=(x + 1, 0.42), fontsize=12, ha="left",
                bbox=dict(boxstyle="round", fc="wheat", alpha=0.85))

ax2.plot(trilha.dia, trilha.efeito, color="tab:blue", lw=2.5, label="efeito máx (PSI)")
ax2.axhline(detector.psi_limiar, color="tab:red", ls="--", lw=2,
            label=f"limiar calibrado ({detector.psi_limiar:.3f})")
ax2.axhline(0.10, color="gray", ls=":", lw=2, label="o 0.1 do folclore")
ax2.set_xlabel("dia"); ax2.set_ylabel("efeito em P(X)")
ax2.legend(loc="upper left", ncol=3)
plt.tight_layout()
plt.show()

# --- a autocorreção honesta, medida ---
base = trilha.query("dia < 59")
tc3 = trilha.query("95 <= dia <= 119")
q_g = base.auc_global.mean() - tc3.auc_global.mean()
q_s = base.auc_cel02.mean() - tc3.auc_cel02.mean()
print(f"queda média no TC-3 — global: {q_g:.3f} | CEL-02: {q_s:.3f} "
      f"({q_s / max(q_g, 1e-9):.1f}× maior)")
print(
    "\n⚠️  É tentador dizer que o monitor global é CEGO ao problema da CEL-02.\n"
    "    Não é verdade — e a primeira pessoa que rodar este notebook te corrige.\n"
    "    Ele TAMBÉM cai. Só que uma fração do tanto, com magnitude que qualquer\n"
    "    engenheiro sênior descartaria como variação amostral — e com razão.\n\n"
    "    A segmentação não transforma o invisível em visível.\n"
    "    Ela transforma o DESCARTÁVEL em ACIONÁVEL."
)

In [ ]:
# ============================================================
#  Camada 4 — traduzindo "a AUC caiu" para a língua da fábrica
# ============================================================
banner("Capabilidade do processo", f"spec: {TORQUE_LSL}–{TORQUE_USL} Nm · alvo Cpk ≥ 1.33", icone="🏭")

cen = [
    ("referência (dias 1–29)", ref),
    ("TC-1 estável", df.query("30 <= dia <= 59")),
    ("TC-3 · CEL-02", df.query("90 <= dia <= 119 and equipamento == 'CEL-02'")),
    ("TC-3 · outras células", df.query("90 <= dia <= 119 and equipamento != 'CEL-02'")),
    ("TC-3' recalibrado", df.query("120 <= dia <= 124 and equipamento == 'CEL-02'")),
]
cap = pd.DataFrame([{
    "cenário": n,
    "média (Nm)": round(d.torque_real.mean(), 2),
    "Cpk": round(cpk(d.torque_real), 3),
    "PPM fora spec": f"{ppm_fora_spec(d.torque_real):,.0f}",
    "conforme?": "✅" if cpk(d.torque_real) >= 1.33 else "❌",
} for n, d in cen])
print(cap.to_string(index=False))

ppm = ppm_fora_spec(cen[2][1].torque_real)
print(f"\n>>> \"a AUC da CEL-02 caiu 0.37\" não para uma fábrica.")
print(f">>> \"{ppm:,.0f} peças por milhão fora de especificação\" para.")
print(">>> Aprenda a traduzir. É a diferença entre ser ouvido e ser ignorado.")

In [ ]:
aposte(
    "Última aposta do bloco, e a que vale o tutorial inteiro. O TC-3 é o caso "
    "MAIS GRAVE: o modelo virou moeda na CEL-02 e o processo está fora de "
    "especificação. Um pipeline maduro, ao ver isso, deveria disparar qual "
    "ação automática?",
    ("retreinar imediatamente", "não fazer nada", "bloquear o retreino"),
)

---
## O clímax: da EVIDÊNCIA para a DECISÃO

Até aqui, tudo que produzimos foi **evidência**: efeito, AUC, Cpk, PPM. Nada
disso é uma decisão. Números não param linha de montagem — **exit codes** param.

A política aplica cinco guardas em cascata. As quatro primeiras são padrão de
mercado e existem para impedir retreino **desnecessário** (blip, sazonalidade,
ruído amostral). A quinta é a contribuição deste tutorial, e existe porque as
outras quatro **falham juntas** num caso específico:

> quando o drift é REAL, PERSISTENTE, de MAGNITUDE ALTA
> e o retreino MELHORA a métrica offline —
> e ainda assim retreinar é a pior decisão possível.

É o TC-3. Retreinar sobre dado de sensor mentiroso **institucionaliza o defeito
mecânico dentro do modelo**. O dashboard fica verde e, quando a metrologia
finalmente calibrar a ferramenta, o modelo quebra — porque ele aprendeu a
mentira como se fosse o normal.

Nenhuma métrica de ML distingue "retreino necessário" de "retreino danoso".
A distinção é **física** — e por isso a guarda 5 é um **bloqueio**, não um score.

In [ ]:
# ============================================================
#  A tabela que o orquestrador consome. Leia a coluna `exit`.
# ============================================================
cli("simular", "--de", "40", "--ate", "130", "--passo", "10")

### O que aconteceu em cada linha

| exit | significa | quem age |
|---|---|---|
| `0` | nada acionável, ou drift com causa conhecida | ninguém |
| `10` | drift real com causa compatível com ML | esteira de retreino |
| `20` | **causa NÃO-ML** — retreino bloqueado | metrologia / engenharia |
| `30` | dado insuficiente para decidir (≠ "sem drift") | quem coleta |

Três coisas para reparar:

1. **O TC-2 nunca sai de `0`.** A campanha de R20 move $$P(X)$$ dezenas de vezes
   o limiar e ainda assim não vira incidente — a performance está intacta e a
   causa é conhecida. É o falso positivo mais caro do mercado, evitado.

2. **O TC-3 sai `20`, não `10`.** É o tutorial inteiro em um número. A causa
   diagnosticada é `hardware`, e `hardware` **não se conserta com GPU**.

3. **O TC-3' volta para `0`.** Depois da recalibração o sistema destrava
   sozinho. Isso importa: política que só sabe dizer "não" é abandonada na
   terceira semana.

> Um pipeline que sabe se **recusar** a retreinar é mais maduro
> que um que retreina rápido.

In [ ]:
# ============================================================
#  O invariante, verificado — não narrado.
# ============================================================
from driftkit.policy import PoliticaRetreino

pol = PoliticaRetreino(detector=detector, piso_aa=PISO_AA, k_de_n=(1, 1))
w110 = janela(df, 110)
d110 = pol.avaliar(
    w110, dia=110, segmento="CEL-02",
    auc_global=metricas(modelo, w110, num=NUM, cat=CAT).roc_auc,
    auc_segmento=metricas(
        modelo, w110.query("equipamento == 'CEL-02'"), num=NUM, cat=CAT
    ).roc_auc,
)
print(d110)
decisao_tc3 = d110.exit_code

assert decisao_tc3 == 20, (
    "o TC-3 DEVE bloquear o retreino. Se este assert falhar, a guarda 5 "
    "regrediu e o argumento do tutorial caiu junto."
)
print("\n✅ TC-3 → exit 20. A ordem de serviço vai para a metrologia, não para a GPU.")

---
## A suíte de regressão — de verdade, com pytest

Tudo que acabamos de verificar à mão vive em `tests/`, como pytest de verdade.
Não é preciosismo:

- `assert` com introspecção (você vê os valores, sem `print`)
- `@pytest.mark.parametrize` roda o controle negativo em **8 seeds** sem duplicar código
- `pytest.approx` distingue **tolerância** de **limiar**
- `@pytest.mark.xfail(strict=True)` documenta **limitação conhecida** — e quebra
  o CI se alguém "consertar" sem atualizar a doc
- **exit code**, que é o que permite virar portão de merge

> Um seed que passa **não é** um teste que passa.

In [ ]:
!cd $RAIZ_REPO && python -m pytest tests/ -q -m "not lento and not requer_bosch and not mercado" --no-header

In [ ]:
# O arquivo mais incomum do repositório — e o de maior valor didático.
# Ele documenta o que o detector NÃO faz.
!cd $RAIZ_REPO && python -m pytest tests/test_limitacoes.py -v --no-header -rxX

Repare nos `XFAIL`. Eles não são falhas — são **limitações provadas**:

| teste | o que documenta |
|---|---|
| `test_referencia_movel_detecta_deriva_lenta` | referência móvel cega o detector a deriva lenta |
| `test_monitor_de_distribuicao_detecta_concept_drift` | $$P(X)$$ **não tem obrigação** de ver concept drift |

> Um monitor honesto sabe dizer onde é cego. Um que não sabe está apenas
> **silencioso** — e silêncio e cegueira produzem o mesmo dashboard verde.

---
## As bibliotecas de mercado (8 min)

Tudo que implementamos à mão existe em biblioteca. **E você deve usar a
biblioteca em produção.** Implementamos do zero por um motivo só: você precisa
entender o instrumento antes de comprá-lo.

A validação cruzada abaixo é o teste de sanidade do nosso código — e, ao mesmo
tempo, a demonstração de que nenhuma dessas ferramentas sabe qual é o **seu**
piso de ruído.

In [ ]:
requer("detector", "df")
w_tc2 = janela(df, 80)

# --- o nosso ---
nosso = detector.report(w_tc2)
print("driftkit (nosso):", nosso.top_features[:4])

# --- alibi-detect ---
try:
    from alibi_detect.cd import KSDrift
    ks = KSDrift(ref[list(NUM)].to_numpy(), p_val=0.05)
    pred = ks.predict(w_tc2[list(NUM)].to_numpy())
    apontadas = [NUM[i] for i, f in enumerate(pred["data"]["is_drift"] if
                 np.ndim(pred["data"]["is_drift"]) else
                 (pred["data"]["p_val"] < 0.05)) if f]
    print("alibi-detect KSDrift  :", apontadas[:4])
except ImportError:
    print("alibi-detect indisponível — sem problema, o argumento não muda")

# --- evidently ---
try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    rep = Report(metrics=[DataDriftPreset()])
    rep.run(reference_data=ref[list(NUM) + list(CAT)],
            current_data=w_tc2[list(NUM) + list(CAT)])
    d = rep.as_dict()["metrics"][0]["result"]
    print(f"evidently             : {d['number_of_drifted_columns']} colunas em drift")
except Exception as e:
    print(f"evidently indisponível ({type(e).__name__}) — o argumento não muda")

print(
    "\n>>> Mesmas features apontadas, mesma ordem de magnitude. Bom sinal.\n"
    ">>> E o ponto que nenhuma delas resolve para você:\n"
    ">>> o default é sempre 0.1 ou p<0.05. NENHUMA sabe qual é o SEU piso.\n"
    ">>> Use a biblioteca. Mas CALIBRE, com o teste A/A que você rodou há 20 min."
)

---
## O contrato v1 — o artefato que atravessa para a Parte II

Salvamos o que aprendemos. Mas repare **o que** transfere e **o que não**:

| MÉTODO (transfere sem alteração) | PARÂMETRO (será remedido) |
|---|---|
| o predicado `p < α/m` **E** `efeito > piso` | o valor do piso |
| as 5 guardas e sua ordem | `k de n`, cooldown |
| as assinaturas diagnósticas | limiar de violação de range |
| a exigência de teste A/A | `min_obs` |

Um limiar calibrado com **9 features, 2.700 amostras/janela e prevalência de
29%** não se aplica a **160 features, 40 mil amostras/janela e prevalência de
0,58%**. Quem transporta o número em vez do método está fazendo chute com
aparência de rigor.

In [ ]:
from driftkit.state import Contrato, salvar_contrato

requer("detector", "PISO_AA", "checks")

contrato_v1 = Contrato(
    versao="v1-sintetico",
    origem=f"fixture Tire & Wheel, seed={SEED}, executada por você agora",
    features={"num": list(NUM), "cat": list(CAT)},
    limiares={
        "psi_piso_aa": PISO_AA,
        "psi_alarme": detector.psi_limiar,
        "alpha": detector.alpha,
        "min_obs": detector.min_obs,
        "bins": detector.bins,
        "sigma_carta": 3.0,
    },
    guardas_retreino={"k_de_n": [3, 5], "cooldown_janelas": 14,
                      "fator_piso": 3.0, "ganho_minimo": 0.01,
                      # guarda 1b: `modelo` é a causa RESIDUAL — o que sobra
                      # quando nenhuma assinatura física fecha — e a única que
                      # AUTORIZA gasto. Por isso é a única que precisa se
                      # repetir para ser aceita.
                      "confirmacao_causa_janelas": 2},
    assinaturas_diagnosticas={
        "pipeline": "violação de range de engenharia > 10% dos registros",
        "hardware": "degradação localizada em segmento com P(X) calmo",
        "negocio": "P(X) em drift com performance preservada",
        "modelo": ("drift persistente sem assinatura física, "
                   "confirmado em 2 janelas consecutivas"),
    },
    ranges={k: list(v) for k, v in RANGES.items()},
    fixture={"features_monitoradas": len(NUM) + len(CAT),
             "amostras_por_janela": int(len(df) / df.dia.nunique() * 3),
             "prevalencia": round(float(ref.falha.mean()), 4), "seed": SEED},
    suite={"pass": len(checks), "total": len(checks)},
)
# Dois nomes, de propósito — e o segundo é o que mata o banner de FALLBACK:
#
#   detector_config.json                ← é o que `carregar_contrato()` procura
#   detector_config_referencia_v1.json  ← é o nome que o CLI grava e que o
#                                         setup baixa como rede de segurança
#
# Enquanto só o segundo existir no disco, o Bloco II avisa que está usando o
# piso da MINHA execução em vez da SUA. Gravar os dois resolve.
salvar_contrato(contrato_v1, DATA / "detector_config.json")
salvar_contrato(contrato_v1, DATA / "detector_config_referencia_v1.json")
print()
print(contrato_v1.resumo())

In [ ]:
checkpoint(
    2,
    fixture_valida=falhas_fixture.empty,
    piso_medido=PISO_AA < 0.05,
    limiar_calibrado=detector.psi_limiar > PISO_AA,
    prova_cruzada=res.query("caso.str.startswith('TC-2')").efeito_max.iloc[0]
                  > res.query("caso.str.startswith('TC-3 ')").efeito_max.iloc[0],
    tc3_bloqueado=(decisao_tc3 == 20),
    contrato_salvo=(DATA / "detector_config.json").exists(),
)

---
# 🎯 Fechamento da Parte I

Você acabou de fazer, com um dataset de brinquedo, três coisas que raramente se
fazem com dados reais:

1. **Mediu o piso de ruído** do seu próprio detector (teste A/A) — e viu que o
   `0.1` da indústria não tem relação com ele.
2. **Provou a diagonal**: o drift que mais move $$P(X)$$ não é o que machuca; o
   que machuca é quase invisível nele.
3. **Documentou as limitações** com testes que falham de propósito.

## ☕ Intervalo técnico (10 min)

Rode a célula abaixo **antes de sair** — ela baixa os dados do Bosch (~180 MB)
enquanto você toma café. Escalonamos de propósito: 40 pessoas puxando 180 MB no
mesmo segundo derruba qualquer wifi de auditório.

In [ ]:
from driftkit.data import baixar_bosch

baixar_bosch()

---
## 🏃 Terminou antes? Escolha um

| | desafio |
|---|---|
| ⭐ | Baixe `psi_limiar` para `0.001` com `detector.com(...)` e capture o TC-1 ficando **vermelho**. Quantas features alarmam? |
| ⭐⭐ | Qual o **menor `offset_torque`** (em Nm) que a suíte ainda detecta? Use `gerar_fixture(offset_torque=...)` e faça a curva de sensibilidade. |
| ⭐⭐ | Troque a métrica: `detector.com(metrica="wasserstein")`. O TC-3 muda de resposta? Por quê a unidade física ajuda a conversar com engenharia? |
| ⭐⭐⭐ | Faça **CEL-02 e CEL-03** derivarem juntas. O monitor segmentado ainda distingue isso de um drift de linha inteira? |
| ⭐⭐⭐ | Rode `detector.calibrar_piso()` com `frac=0.1`. O piso muda? O que isso diz sobre janelas pequenas? |

---

### ➡️ Próximo: `02_real.ipynb`

> Parte I: *"meu detector acerta?"*

> Parte II: **"e quando ninguém me diz a resposta?"**